# Воркшоп №2 — Эмбеддинги модальностей

**Курсовая:** «Использование мультимодальных эмбеддингов в рекомендательных системах».

После воркшопа №1 у меня есть `metadata.parquet`, аудио, обложки и `edges.parquet`.

Здесь я считаю эмбеддинги для каждой модальности:

| Модальность | Модель | Размерность |
|---|---|---|
| Текст | `intfloat/multilingual-e5-base` | 768 |
| Изображение | `openai/clip-vit-base-patch32` | 512 |
| Аудио | `laion/clap-htsat-unfused` | 512 |

Сохраняю в `data/embeddings/*.npz` с общим массивом `track_ids`.

## 0. Установка зависимостей

In [1]:
%pip install -q torch torchvision transformers sentence-transformers pillow librosa soundfile audioread numpy pandas tqdm pyarrow

Note: you may need to restart the kernel to use updated packages.


## 1. Импорты и конфиг

Загружаю модели и выбираю устройство (CPU или MPS на Mac).

In [2]:
from __future__ import annotations

import os
import gc
import time
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("embeddings")

DATA_DIR  = Path("data")
EMB_DIR   = DATA_DIR / "embeddings"
EMB_DIR.mkdir(parents=True, exist_ok=True)

META_PATH  = DATA_DIR / "metadata.parquet"
TEXT_PATH  = EMB_DIR  / "text.npz"
IMAGE_PATH = EMB_DIR  / "image.npz"
AUDIO_PATH = EMB_DIR  / "audio.npz"


def pick_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


DEVICE = pick_device()
log.info("Using device: %s", DEVICE)

TEXT_MODEL_ID  = "intfloat/multilingual-e5-base"
IMAGE_MODEL_ID = "openai/clip-vit-base-patch32"
AUDIO_MODEL_ID = "laion/clap-htsat-unfused"

2026-06-09 20:05:32,559 | INFO | Using device: mps


## 2. Загрузка датасета

Беру треки, у которых есть все три модальности. Получилось **14 787** треков.

In [3]:
df_all = pd.read_parquet(META_PATH)
log.info("Total tracks in metadata: %d", len(df_all))


def _path_exists(p) -> bool:
    if not isinstance(p, str) or not p:
        return False
    return Path(p).exists()


mask = (
    df_all["has_audio"].fillna(False).astype(bool)
    & df_all["has_image"].fillna(False).astype(bool)
    & df_all["audio_path"].apply(_path_exists)
    & df_all["image_path"].apply(_path_exists)
)
df = df_all.loc[mask].copy().reset_index(drop=True)
df["track_id"] = df["track_id"].astype(np.int64)

log.info("Tracks with audio+image present on disk: %d (%.1f%%)",
         len(df), 100 * len(df) / max(len(df_all), 1))

TRACK_IDS = df["track_id"].to_numpy()
print("Кол-во треков в эмбеддинг-сете:", len(TRACK_IDS))
print("Первые 5 track_id:", TRACK_IDS[:5].tolist())

2026-06-09 20:05:32,722 | INFO | Total tracks in metadata: 16899


2026-06-09 20:05:32,857 | INFO | Tracks with audio+image present on disk: 14787 (87.5%)


Кол-во треков в эмбеддинг-сете: 14787
Первые 5 track_id: [1772783961, 1358462132, 1411628233, 617154366, 1440813515]


## 3. Текстовые эмбеддинги (E5)

Считаю эмбеддинги по `text_blob` моделью `multilingual-e5-base`.

In [4]:
def compute_text_embeddings(texts: list[str], *, batch_size: int = 32) -> np.ndarray:
    from transformers import AutoTokenizer, AutoModel

    tok = AutoTokenizer.from_pretrained(TEXT_MODEL_ID)
    model = AutoModel.from_pretrained(TEXT_MODEL_ID).to(DEVICE).eval()

    prefixed = [f"passage: {t or ''}" for t in texts]
    all_emb: list[np.ndarray] = []
    with torch.no_grad():
        for i in tqdm(range(0, len(prefixed), batch_size), desc="text"):
            batch = prefixed[i : i + batch_size]
            enc = tok(batch, padding=True, truncation=True, max_length=512, return_tensors="pt")
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            out = model(**enc)
            mask = enc["attention_mask"].unsqueeze(-1).float()
            summed = (out.last_hidden_state * mask).sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1)
            mean = summed / counts
            mean = torch.nn.functional.normalize(mean, p=2, dim=1)
            all_emb.append(mean.cpu().to(torch.float32).numpy())

    del model, tok
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()
    elif DEVICE == "cuda":
        torch.cuda.empty_cache()
    return np.concatenate(all_emb, axis=0)


if TEXT_PATH.exists():
    log.info("Text embeddings already exist at %s — skipping.", TEXT_PATH)
    npz = np.load(TEXT_PATH)
    text_emb = npz["emb"]
    assert (npz["track_ids"] == TRACK_IDS).all(), "Кэш text.npz не совпадает с текущим набором track_ids — удали файл и перезапусти."
else:
    t0 = time.time()
    text_emb = compute_text_embeddings(df["text_blob"].fillna("").tolist())
    np.savez_compressed(TEXT_PATH, track_ids=TRACK_IDS, emb=text_emb.astype(np.float32))
    log.info("Text embeddings: shape=%s, saved to %s, %.1fs",
             text_emb.shape, TEXT_PATH, time.time() - t0)

print("text_emb shape:", text_emb.shape)
print("text_emb dtype:", text_emb.dtype)
print("L2-нормы (первые 5):", np.linalg.norm(text_emb[:5], axis=1).round(4))

2026-06-09 20:05:32,863 | INFO | Text embeddings already exist at data/embeddings/text.npz — skipping.


text_emb shape: (14787, 768)
text_emb dtype: float32
L2-нормы (первые 5): [1. 1. 1. 1. 1.]


## 4. Визуальные эмбеддинги (CLIP)

In [5]:
def _safe_open_image(path: str | Path) -> Image.Image:
    img = Image.open(path)
    if img.mode != "RGB":
        img = img.convert("RGB")
    return img


def _as_tensor(out) -> torch.Tensor:
    """В transformers 5.x get_image_features / get_audio_features могут вернуть
    BaseModelOutputWithPooling вместо тензора. Достаём фактический эмбеддинг."""
    if isinstance(out, torch.Tensor):
        return out
    for attr in ("image_embeds", "audio_embeds", "pooler_output", "last_hidden_state"):
        v = getattr(out, attr, None)
        if isinstance(v, torch.Tensor):
            if v.dim() == 3:
                v = v.mean(dim=1)
            return v
    raise TypeError(f"Cannot extract tensor from {type(out).__name__}")


def compute_image_embeddings(paths: list[str], *, batch_size: int = 32) -> np.ndarray:
    from transformers import CLIPModel, CLIPProcessor

    proc = CLIPProcessor.from_pretrained(IMAGE_MODEL_ID)
    model = CLIPModel.from_pretrained(IMAGE_MODEL_ID).to(DEVICE).eval()

    all_emb: list[np.ndarray] = []
    with torch.no_grad():
        for i in tqdm(range(0, len(paths), batch_size), desc="image"):
            chunk = paths[i : i + batch_size]
            images = []
            for p in chunk:
                try:
                    images.append(_safe_open_image(p))
                except Exception as e:
                    log.warning("Bad image %s: %s", p, e)
                    images.append(Image.new("RGB", (224, 224)))
            enc = proc(images=images, return_tensors="pt")
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            feats = _as_tensor(model.get_image_features(**enc))
            feats = torch.nn.functional.normalize(feats, p=2, dim=1)
            all_emb.append(feats.cpu().to(torch.float32).numpy())

    del model, proc
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()
    elif DEVICE == "cuda":
        torch.cuda.empty_cache()
    return np.concatenate(all_emb, axis=0)


if IMAGE_PATH.exists():
    log.info("Image embeddings already exist at %s — skipping.", IMAGE_PATH)
    npz = np.load(IMAGE_PATH)
    image_emb = npz["emb"]
    assert (npz["track_ids"] == TRACK_IDS).all(), "Кэш image.npz не совпадает — удали файл и перезапусти."
else:
    t0 = time.time()
    image_emb = compute_image_embeddings(df["image_path"].tolist())
    np.savez_compressed(IMAGE_PATH, track_ids=TRACK_IDS, emb=image_emb.astype(np.float32))
    log.info("Image embeddings: shape=%s, saved to %s, %.1fs",
             image_emb.shape, IMAGE_PATH, time.time() - t0)

print("image_emb shape:", image_emb.shape)
print("image_emb dtype:", image_emb.dtype)
print("L2-нормы (первые 5):", np.linalg.norm(image_emb[:5], axis=1).round(4))

2026-06-09 20:05:37,003 | INFO | HTTP Request: GET https://huggingface.co/api/models/openai/clip-vit-base-patch32/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"


2026-06-09 20:05:37,227 | INFO | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"


2026-06-09 20:05:37,229 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


2026-06-09 20:05:37,450 | INFO | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"


2026-06-09 20:05:37,683 | INFO | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"


...

image:   0%|          | 0/463 [00:00<?, ?it/s]

2026-06-09 20:05:43,098 | INFO | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/refs%2Fpr%2F66/model.safetensors.index.json "HTTP/1.1 404 Not Found"


2026-06-09 20:05:43,320 | INFO | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/refs%2Fpr%2F66/model.safetensors "HTTP/1.1 302 Found"


2026-06-09 20:08:46,801 | INFO | Image embeddings: shape=(14787, 512), saved to data/embeddings/image.npz, 193.8s


image_emb shape: (14787, 512)
image_emb dtype: float32
L2-нормы (первые 5): [1. 1. 1. 1. 1.]


## 5. Аудио-эмбеддинги (CLAP)

In [6]:
CLAP_SR = 48000

import subprocess
import imageio_ffmpeg

_FFMPEG_EXE = imageio_ffmpeg.get_ffmpeg_exe()


def _load_audio(path: str | Path, *, sr: int = CLAP_SR, max_seconds: float = 30.0) -> np.ndarray:
    """Декодирование m4a в mono через ffmpeg."""
    cmd = [
        _FFMPEG_EXE, "-v", "quiet",
        "-i", str(path),
        "-t", f"{max_seconds:.3f}",
        "-f", "f32le",
        "-ac", "1",
        "-ar", str(sr),
        "-",
    ]
    proc = subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    audio = np.frombuffer(proc.stdout, dtype=np.float32).copy()
    if audio.size == 0:
        raise RuntimeError(f"empty audio: {path}")
    return audio


def compute_audio_embeddings(paths: list[str], *, batch_size: int = 8) -> tuple[np.ndarray, list[int]]:
    from transformers import ClapModel, ClapProcessor

    proc = ClapProcessor.from_pretrained(AUDIO_MODEL_ID)
    model = ClapModel.from_pretrained(AUDIO_MODEL_ID).to(DEVICE).eval()

    all_emb: list[np.ndarray] = []
    bad_idx: list[int] = []
    EMB_DIM = 512

    with torch.no_grad():
        for i in tqdm(range(0, len(paths), batch_size), desc="audio"):
            chunk = paths[i : i + batch_size]
            wavs: list[np.ndarray] = []
            local_bad: list[int] = []
            for j, p in enumerate(chunk):
                try:
                    wavs.append(_load_audio(p))
                except Exception as e:
                    log.warning("Bad audio %s: %s", p, e)
                    local_bad.append(j)
                    wavs.append(np.zeros(CLAP_SR * 10, dtype=np.float32))

            try:
                enc = proc(audio=wavs, sampling_rate=CLAP_SR, return_tensors="pt", padding=True)
                enc = {k: v.to(DEVICE) for k, v in enc.items()}
                feats = _as_tensor(model.get_audio_features(**enc))
                feats = torch.nn.functional.normalize(feats, p=2, dim=1)
                feats_np = feats.cpu().to(torch.float32).numpy()
            except Exception as e:
                log.warning("CLAP batch fail at i=%d: %s — filling zeros", i, e)
                feats_np = np.zeros((len(chunk), EMB_DIM), dtype=np.float32)
                local_bad = list(range(len(chunk)))

            for j in local_bad:
                feats_np[j] = 0.0
                bad_idx.append(i + j)

            all_emb.append(feats_np)

    del model, proc
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()
    elif DEVICE == "cuda":
        torch.cuda.empty_cache()
    return np.concatenate(all_emb, axis=0), bad_idx


if AUDIO_PATH.exists():
    log.info("Audio embeddings already exist at %s — skipping.", AUDIO_PATH)
    npz = np.load(AUDIO_PATH)
    audio_emb = npz["emb"]
    assert (npz["track_ids"] == TRACK_IDS).all(), "Кэш audio.npz не совпадает — удали файл и перезапусти."
    bad_audio_idx: list[int] = []
else:
    t0 = time.time()
    audio_emb, bad_audio_idx = compute_audio_embeddings(df["audio_path"].tolist(), batch_size=8)
    np.savez_compressed(AUDIO_PATH, track_ids=TRACK_IDS, emb=audio_emb.astype(np.float32))
    log.info("Audio embeddings: shape=%s, %d bad files, saved to %s, %.1fs",
             audio_emb.shape, len(bad_audio_idx), AUDIO_PATH, time.time() - t0)

print("audio_emb shape:", audio_emb.shape)
print("audio_emb dtype:", audio_emb.dtype)
print("L2-нормы (первые 5):", np.linalg.norm(audio_emb[:5], axis=1).round(4))
if bad_audio_idx:
    print(f"Битых аудио: {len(bad_audio_idx)} ({len(bad_audio_idx) / len(audio_emb):.1%})")

2026-06-09 20:08:51,082 | INFO | HTTP Request: GET https://huggingface.co/api/models/laion/clap-htsat-unfused/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"


2026-06-09 20:08:51,312 | INFO | HTTP Request: HEAD https://huggingface.co/laion/clap-htsat-unfused/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"


2026-06-09 20:08:51,528 | INFO | HTTP Request: HEAD https://huggingface.co/laion/clap-htsat-unfused/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"


2026-06-09 20:08:51,747 | INFO | HTTP Request: HEAD https://huggingface.co/laion/clap-htsat-unfused/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"


2026-06-09 20:08:51,970 | INFO | HTTP Request: HEAD https://huggingface.co/laion/clap-htsat-unfused/resolve/main/audio_tokenizer_config.json "HTTP/1.1 404 Not Found"


...

2026-06-09 20:08:57,209 | INFO | HTTP Request: HEAD https://huggingface.co/laion/clap-htsat-unfused/resolve/refs%2Fpr%2F3/model.safetensors.index.json "HTTP/1.1 404 Not Found"


2026-06-09 20:08:57,435 | INFO | HTTP Request: HEAD https://huggingface.co/laion/clap-htsat-unfused/resolve/refs%2Fpr%2F3/model.safetensors "HTTP/1.1 302 Found"


audio:   0%|          | 0/1849 [00:00<?, ?it/s]

2026-06-09 20:38:23,023 | INFO | Audio embeddings: shape=(14787, 512), 0 bad files, saved to data/embeddings/audio.npz, 1772.8s


audio_emb shape: (14787, 512)
audio_emb dtype: float32
L2-нормы (первые 5): [1. 1. 1. 1. 1.]


## 6. Проверка ближайших соседей

Смотрю, какие треки оказываются близкими по каждой модальности — чтобы убедиться, что эмбеддинги осмысленные.

In [7]:
def top_k_neighbors(emb: np.ndarray, idx: int, k: int = 5) -> list[tuple[int, float]]:
    sims = emb @ emb[idx]
    sims[idx] = -np.inf
    top = np.argpartition(-sims, k)[:k]
    top = top[np.argsort(-sims[top])]
    return [(int(i), float(sims[i])) for i in top]


def show_neighbors(idx: int, k: int = 5) -> None:
    row = df.iloc[idx]
    print(f"=== Anchor: [{row['track_id']}] {row['artist']} — {row['title']}  "
          f"(genre: {row['genre_itunes']})")
    print()
    for name, emb in [("TEXT", text_emb), ("IMAGE", image_emb), ("AUDIO", audio_emb)]:
        print(f"--- {name} ---")
        for n_idx, score in top_k_neighbors(emb, idx, k=k):
            r = df.iloc[n_idx]
            print(f"  {score:+.3f}  [{r['track_id']:>10}]  {r['artist']} — {r['title']}  ({r['genre_itunes']})")
        print()


N_PROBES = 3
rng = np.random.default_rng(seed=42)
probe_idx = rng.choice(len(df), size=N_PROBES, replace=False)
for i in probe_idx:
    show_neighbors(int(i), k=5)
    print("=" * 88)

=== Anchor: [1538367584] Dj Vibes EDM — EDM Party Dance  (genre: Ambient)

--- TEXT ---


  +0.991  [1538367594]  Dj Vibes EDM — Halloween Dubstep  (Ambient)
  +0.986  [1538367586]  Dj Vibes EDM — Halloween Club  (Ambient)
  +0.984  [1538367591]  Dj Vibes EDM — Drum and Bass  (Ambient)
  +0.978  [1538367592]  Dj Vibes EDM — Dark Night  (Ambient)
  +0.977  [1538367593]  Dj Vibes EDM — Zombies Chill  (Ambient)

--- IMAGE ---


  +1.000  [1538367587]  Dj Vibes EDM — Atmosphere of Fear  (Ambient)
  +1.000  [1538367596]  Dj Vibes EDM — Spooky Night  (Ambient)
  +1.000  [1538367597]  Dj Vibes EDM — Highway to Hell  (Ambient)
  +1.000  [1538367588]  Dj Vibes EDM — Dance with the Demons  (Ambient)
  +1.000  [1538367590]  Dj Vibes EDM — In the Darkness  (Ambient)

--- AUDIO ---
  +0.949  [1538367587]  Dj Vibes EDM — Atmosphere of Fear  (Ambient)
  +0.938  [1538367588]  Dj Vibes EDM — Dance with the Demons  (Ambient)
  +0.935  [1538367597]  Dj Vibes EDM — Highway to Hell  (Ambient)
  +0.929  [ 389778578]  Yakooza — Cocaine (Scot Project Remix)  (Dance)
  +0.909  [ 393138850]  Autoerotique — Bubonic  (Electronic)

=== Anchor: [1066113456] What So Not — Gemini (feat. George Maple)  (genre: Hip-Hop)

--- TEXT ---
  +0.914  [1709039068]  George Maple — Talk Talk  (Electronic)
  +0.907  [1697404704]  RL Grime & What So Not — Tell Me  (Electronic)
  +0.907  [1054512450]  ZHU x AlunaGeorge — Automatic  (Dance)
  +0.903  [ 

  +0.672  [1440773792]  Bobby Brown — Every Little Step  (Pop)
  +0.672  [1440773584]  Bobby Brown — My Prerogative  (Pop)
  +0.668  [1443564076]  Brian McKnight — One Last Cry  (R&B/Soul)
  +0.665  [ 326644034]  Iyaz — Replay  (Pop)
  +0.661  [ 724026818]  O'Bryan — Lovelite  (R&B/Soul)

--- AUDIO ---
  +0.948  [ 440309582]  J Boog — Let's Do It Again  (Reggae)
  +0.946  [1163187623]  Shabba Ranks — Twice My Age (feat. Krystal)  (Brazilian)
  +0.944  [ 724453907]  Shaggy — Big Up  (Reggae)
  +0.943  [ 309082433]  Half Pint — Substitute Lover  (Reggae)
  +0.939  [ 417056588]  Duncan Mighty — Obianuju  (Reggae)



### 6.1 Recall@10 против Last.fm

Считаю Recall@10 относительно соседей из `edges.parquet`. Это предварительная проверка перед воркшопом №4.

In [8]:
EDGES_PATH = DATA_DIR / "edges.parquet"
edges_df = pd.read_parquet(EDGES_PATH)
edges_in = edges_df.loc[edges_df["in_corpus"] & edges_df["dst_track_id"].notna()].copy()
edges_in["dst_track_id"] = edges_in["dst_track_id"].astype(np.int64)

tid_to_idx = {int(t): i for i, t in enumerate(TRACK_IDS)}
edges_in = edges_in.loc[
    edges_in["src_track_id"].isin(tid_to_idx) & edges_in["dst_track_id"].isin(tid_to_idx)
]
log.info("Edges usable for sanity-check: %d (src tracks: %d)",
         len(edges_in), edges_in["src_track_id"].nunique())

K_GT   = 10
K_PRED = 10

gt_by_src: dict[int, set[int]] = {}
for src_tid, grp in edges_in.sort_values("match", ascending=False).groupby("src_track_id"):
    top = grp.head(K_GT)["dst_track_id"].astype(int).tolist()
    if top:
        gt_by_src[int(src_tid)] = set(top)


def recall_at_k(emb: np.ndarray, gt_by_src: dict[int, set[int]], k: int = K_PRED) -> float:
    hits = 0
    total = 0
    src_idxs = [tid_to_idx[s] for s in gt_by_src if s in tid_to_idx]
    chunk = 256
    for start in range(0, len(src_idxs), chunk):
        block = src_idxs[start : start + chunk]
        sims = emb[block] @ emb.T
        for j, i in enumerate(block):
            sims[j, i] = -np.inf
        top_idx = np.argpartition(-sims, k, axis=1)[:, :k]
        for row_j, src_i in enumerate(block):
            src_tid = int(TRACK_IDS[src_i])
            gt = gt_by_src[src_tid]
            pred_tids = {int(TRACK_IDS[i]) for i in top_idx[row_j]}
            hits += len(pred_tids & gt) / max(len(gt), 1)
            total += 1
    return hits / max(total, 1)


def random_recall(gt_by_src: dict[int, set[int]], k: int = K_PRED) -> float:
    N = len(TRACK_IDS)
    eps = []
    for _, gt in gt_by_src.items():
        eps.append(min(k * len(gt) / (N - 1), 1.0))
    return float(np.mean(eps))


print("Sanity-check: Recall@10 vs Last.fm getSimilar (top-10 ground truth)")
print(f"  TEXT  : {recall_at_k(text_emb,  gt_by_src):.3f}")
print(f"  IMAGE : {recall_at_k(image_emb, gt_by_src):.3f}")
print(f"  AUDIO : {recall_at_k(audio_emb, gt_by_src):.3f}")
print(f"  RANDOM: {random_recall(gt_by_src):.4f}  (теоретический baseline)")

2026-06-09 20:38:26,735 | INFO | Edges usable for sanity-check: 81855 (src tracks: 7606)


Sanity-check: Recall@10 vs Last.fm getSimilar (top-10 ground truth)


  TEXT  : 0.080


  IMAGE : 0.075


  AUDIO : 0.025
  RANDOM: 0.0047  (теоретический baseline)


## 7. Итог

Сохранил `data/embeddings/text.npz`, `image.npz`, `audio.npz`. Дальше — fusion в `03_fusion.ipynb`.